# Practice - 08 - Lakeflow Jobs: branching, loops, retries, and triggers

Prerequisites / execution context: Run 00 and 01. Use the bundle sources for actual job tasks. Jobs task values require a job run context. This notebook provides setup and verification instructions.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R37: Lakeflow Jobs](https://docs.databricks.com/aws/en/jobs/); [R38: If/else tasks](https://docs.databricks.com/aws/en/jobs/tasks/if-else); [R39: For each tasks](https://docs.databricks.com/aws/en/jobs/tasks/for-each); [R40: Dynamic value references](https://docs.databricks.com/aws/en/jobs/dynamic-value-references); [R41: Task values](https://docs.databricks.com/aws/en/jobs/task-values); [R42: Job triggers](https://docs.databricks.com/aws/en/jobs/triggers); [R43: Configure job tasks and timeouts](https://docs.databricks.com/aws/en/jobs/configure-task).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Written / design task 1

Design quality -> choose_join -> exclusive left/inner branches -> publish -> per_country. Require a join_type parameter; publish after at least one successful branch; process IT and FR in a For each task. Explain required branch outcome dependencies. Draft the DAG and settings below.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMS4gQnVpbGQgb3IgZGVwbG95IHRoZSBqb2IKRGVwbG95IGJ1bmRsZS9kYXRhYnJpY2tzLnltbCB1c2luZyBsYWIgMDksIG9yIGNyZWF0ZSB0aGUgc2FtZSB0YXNrcyBpbiB0aGUgVUkuIFRoZSBwYXJhbWV0ZXIgam9pbl90eXBlIGRlZmF1bHRzIHRvIGxlZnQuIFRhc2tzOiBxdWFsaXR5IC0mZ3Q7IGNob29zZV9qb2luIC0mZ3Q7IGxlZnRfam9pbiBvbiB0cnVlIC8gaW5uZXJfam9pbiBvbiBmYWxzZSAtJmd0OyBwdWJsaXNoIHdoZW4gYXQgbGVhc3Qgb25lIGJyYW5jaCBzdWNjZWVkcyAtJmd0OyBwZXJfY291bnRyeSBsb29wLiBVc2UgdGhlIGV4YWN0IGJyYW5jaCBvdXRjb21lIGRlcGVuZGVuY3ksIG5vdCBvbmx5IGEgcGxhaW4gZGVwZW5kZW5jeSBvbiBjaG9vc2Vfam9pbi4KPC9kaXY+PC9kZXRhaWxzPg==')


## Written / design task 2

Write the reference syntax for a job parameter, an upstream task-produced value, and a bundle deployment variable. Explain where each is evaluated.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDIgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMi4gVGhyZWUgbmFtZXNwYWNlcwpKb2IgcGFyYW1ldGVyOiB7e2pvYi5wYXJhbWV0ZXJzLmpvaW5fdHlwZX19LiBUYXNrLXByb2R1Y2VkIHZhbHVlOiB7e3Rhc2tzLnF1YWxpdHkudmFsdWVzLnZhbGlkX3Jvd3N9fS4gQnVuZGxlIGRlcGxveW1lbnQgdmFyaWFibGU6ICR7dmFyLmNhdGFsb2d9LiBEeW5hbWljIHJlZmVyZW5jZXMgYXJlIHJlc29sdmVkIGluIGpvYiBjb25maWd1cmF0aW9uIGFuZCBwYXNzZWQgaW50byBub3RlYm9vayBwYXJhbWV0ZXJzOyB3cml0aW5nIHRob3NlIGJyYWNlcyBpbnNpZGUgb3JkaW5hcnkgUHl0aG9uIGRvZXMgbm90IHJlc29sdmUgdGhlbS4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2pvYnMvZHluYW1pYy12YWx1ZS1yZWZlcmVuY2VzIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjQwOiBEeW5hbWljIHZhbHVlIHJlZmVyZW5jZXM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2Rldi10b29scy9idW5kbGVzL3ZhcmlhYmxlcyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI0ODogQnVuZGxlIHZhcmlhYmxlczwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')


## Coding task 1: Verify your branch runs

After your left and inner runs, inspect GOLD.job_join_result. Record output count for each, the selected branch, and the condition expression evaluated by Jobs. Use an opt-in check.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
CHECK_JOB = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIFJ1biB0d2ljZSBhbmQgdmVyaWZ5CkZpcnN0IHJ1biBqb2luX3R5cGU9bGVmdDogbGVmdCBicmFuY2ggZXhlY3V0ZXMgYW5kIGlubmVyIGJyYW5jaCBpcyBleGNsdWRlZDsgb3V0cHV0IGhhcyA0IHJvd3MuIFNlY29uZCBydW4gam9pbl90eXBlPWlubmVyOiBpbm5lciBicmFuY2ggZXhlY3V0ZXMgYW5kIGxlZnQgaXMgZXhjbHVkZWQ7IG91dHB1dCBoYXMgMiByb3dzLiBJbiB0aGUgcnVuIGRldGFpbHMgaW5zcGVjdCB0aGUgZXZhbHVhdGVkIGNvbmRpdGlvbiBhbmQgYWN0dWFsIHRhc2sgcGFyYW1ldGVycy4gQ29tcGFyZSBTVFJJTkcgdmFsdWVzIGV4YWN0bHk6IGxlZnQgdmVyc3VzIExlZnQgZGlmZmVyLiBUaGUgdHdvIGJyYW5jaGVzIHdyaXRlIHRoZSBzYW1lIHRhcmdldCBzYWZlbHkgYmVjYXVzZSBleGFjdGx5IG9uZSBicmFuY2ggcnVucyBhbmQgbWF4X2NvbmN1cnJlbnRfcnVucyBpcyAxLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPiMgUnVuIGFmdGVyIGVhY2ggam9iLCBmcm9tIHRoaXMgbm90ZWJvb2s7IGpvYiBzb3VyY2VzIHdyaXRlIHRvIGEgc2VwYXJhdGUgdGFyZ2V0LgpDSEVDS19KT0I9RmFsc2UKaWYgQ0hFQ0tfSk9COgogICAgb3V0PXNwYXJrLnRhYmxlKGYmcXVvdDt7R09MRH0uam9iX2pvaW5fcmVzdWx0JnF1b3Q7KQogICAgb3V0Lm9yZGVyQnkoJnF1b3Q7b3JkZXJfaWQmcXVvdDspLnNob3coKQogICAgYXNzZXJ0IG91dC5jb3VudCgpIGluICgyLDQpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 3

Create a failed quality check in a development copy, inspect downstream task states, repair the defect and compare retry versus repair behavior. Distinguish timeout, warning threshold and compute auto-termination.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDMgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNC4gRmFpbHVyZSBhbmQgcmVwYWlyIGV4ZXJjaXNlClRlbXBvcmFyaWx5IG1ha2UgdGhlIHF1YWxpdHkgdGFzayBhc3NlcnRpb24gZmFpbCBpbiBhIGRldmVsb3BtZW50IGNvcHkuIE9ic2VydmUgdXBzdHJlYW0gZmFpbHVyZSBhbmQgYmxvY2tlZCBkb3duc3RyZWFtIHRhc2tzLiBSZXBhaXIgYWZ0ZXIgZml4aW5nIHRoZSBkZWZlY3QgYW5kIGluc3BlY3Qgd2hpY2ggdGFza3MgcmVydW4uIFJldHJpZXMgc3VpdCB0cmFuc2llbnQgZXJyb3JzOyBpbnZhbGlkIGRhdGEgbmVlZHMgYSBwb2xpY3kuIENvbmZpZ3VyZSBhIDkwMC1zZWNvbmQgdGFzayB0aW1lb3V0IGFuZCBkaXN0aW5ndWlzaCBpdCBmcm9tIGEgd2FybmluZyB0aHJlc2hvbGQgYW5kIGNsdXN0ZXIgYXV0by10ZXJtaW5hdGlvbi4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2pvYnMvY29uZmlndXJlLXRhc2siIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNDM6IENvbmZpZ3VyZSBqb2IgdGFza3MgYW5kIHRpbWVvdXRzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL21vbml0b3IiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNDQ6IE1vbml0b3IgSm9iczwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')


## Written / design task 4

Select a trigger for fixed reporting time, unpredictable files and changed source tables. Add SQL, pipeline and dashboard tasks in a development copy; identify compute and permissions requirements.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDQgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNS4gVHJpZ2dlciBzZWxlY3Rpb24gYW5kIGV4dHJhIHRhc2sgdHlwZXMKVXNlIGEgc2NoZWR1bGUgZm9yIGEgZml4ZWQgcmVwb3J0aW5nIHRpbWUsIGZpbGUgYXJyaXZhbCBmb3IgdW5wcmVkaWN0YWJsZSBmaWxlIGRlbGl2ZXJ5LCBhbmQgdGFibGUgdXBkYXRlIGZvciBjaGFuZ2VkIHVwc3RyZWFtIHRhYmxlcy4gQ29uZmlybSBwYXRoIHBlcm1pc3Npb25zIGFuZCBmZWF0dXJlIGF2YWlsYWJpbGl0eS4gSW4gYSBkZXZlbG9wbWVudCBjb3B5IGFkZCBhIFNRTCB0YXNrIHF1ZXJ5aW5nIHRoZSBHb2xkIHRhYmxlIChzZWxlY3QgYSBTUUwgd2FyZWhvdXNlKSwgYSBwaXBlbGluZSB0YXNrIHJlZmVyZW5jaW5nIGxhYiAwNywgYW5kIGEgZGFzaGJvYXJkIHRhc2sgcmVmZXJlbmNpbmcgYSBzYXZlZCBkYXNoYm9hcmQuIFNldCBkZXBlbmRlbmNpZXMgc28gZGFzaGJvYXJkIHJlZnJlc2ggZm9sbG93cyBzdWNjZXNzZnVsIEdvbGQgcHVibGljYXRpb24uIFJlY29yZCB0aGVpciBjb21wdXRlIGFuZCBwZXJtaXNzaW9ucyByZXF1aXJlbWVudHMuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL3RyaWdnZXJzIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjQyOiBKb2IgdHJpZ2dlcnM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2pvYnMvY29uZmlndXJlLXRhc2siIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNDM6IENvbmZpZ3VyZSBqb2IgdGFza3MgYW5kIHRpbWVvdXRzPC9hPi4KPC9kaXY+PC9kZXRhaWxzPg==')


## Written / design task 5

Describe For each inputs and the parameter passed to the nested notebook. Choose a concurrency for a small account and explain nesting limitations.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDUgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNi4gTG9vcCBjaGVjawpUaGUgYnVuZGxlIEZvciBlYWNoIGlucHV0cyBjb250YWluIElUIGFuZCBGUjsgdGhlIG5lc3RlZCBub3RlYm9vayByZWNlaXZlcyB7e2lucHV0LmNvdW50cnl9fSBhbmQgd3JpdGVzIHNlcGFyYXRlIGNvdW50cnkgdGFibGVzLiBDb25jdXJyZW5jeSBpcyAxIGZvciBhIHNtYWxsIGFjY291bnQuIEEgRm9yIGVhY2ggdGFzayBjYW5ub3QgZGlyZWN0bHkgbmVzdCBhbm90aGVyIEZvciBlYWNoLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy90YXNrcy9mb3ItZWFjaCIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIzOTogRm9yIGVhY2ggdGFza3M8L2E+Lgo8L2Rpdj48L2RldGFpbHM+')


## Optional supporting reference files
Try writing your own configuration/source first. These panels contain the full source needed for the external pipeline/Job tasks. Reveal, then save the code to the stated workspace file paths. They are not executed by opening a panel. Replace catalog/prefix settings with your practice configuration.


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgSm9iIERBRyBjb25maWd1cmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9kZXYtdG9vbHMvYnVuZGxlcy9zZXR0aW5ncwojIEFkZCB3b3Jrc3BhY2UuaG9zdCB0byBlYWNoIHRhcmdldCwgb3IgdXNlIGEgY29ycmVjdGx5IGNvbmZpZ3VyZWQgYXV0aGVudGljYXRpb24gY29udGV4dC4KIyBTZXJ2ZXJsZXNzIGpvYjogbm8gZXhwbGljaXQgY2xhc3NpYyBjbHVzdGVyLiBObyBzY2hlZHVsZSBpcyBlbmFibGVkLgpidW5kbGU6CiAgbmFtZTogZGVhLXN0dWR5LWxha2Vob3VzZQp2YXJpYWJsZXM6CiAgY2F0YWxvZzoKICAgIGRlc2NyaXB0aW9uOiBFeGlzdGluZyB3cml0YWJsZSBVbml0eSBDYXRhbG9nIGNhdGFsb2cKICAgIGRlZmF1bHQ6IHdvcmtzcGFjZQogIHByZWZpeDoKICAgIGRlc2NyaXB0aW9uOiBJc29sYXRlZCBsYWIgc2NoZW1hIHByZWZpeAogICAgZGVmYXVsdDogZGVhX3JpY2NhcmRvX3ByYWN0aWNlCnJlc291cmNlczoKICBqb2JzOgogICAgb3JkZXJzX2pvYjoKICAgICAgbmFtZTogZGVhLW9yZGVycy0ke2J1bmRsZS50YXJnZXR9CiAgICAgIG1heF9jb25jdXJyZW50X3J1bnM6IDEKICAgICAgcGFyYW1ldGVyczoKICAgICAgLSBuYW1lOiBqb2luX3R5cGUKICAgICAgICBkZWZhdWx0OiBsZWZ0CiAgICAgIHRhc2tzOgogICAgICAtIHRhc2tfa2V5OiBxdWFsaXR5CiAgICAgICAgbm90ZWJvb2tfdGFzazoKICAgICAgICAgIG5vdGVib29rX3BhdGg6IC4vc3JjL3F1YWxpdHkucHkKICAgICAgICAgIGJhc2VfcGFyYW1ldGVyczoKICAgICAgICAgICAgY2F0YWxvZzogJHt2YXIuY2F0YWxvZ30KICAgICAgICAgICAgcHJlZml4OiAke3Zhci5wcmVmaXh9CiAgICAgICAgICAgIGpvaW5fdHlwZTogJiN4Mjc7e3tqb2IucGFyYW1ldGVycy5qb2luX3R5cGV9fSYjeDI3OwogICAgICAgIHRpbWVvdXRfc2Vjb25kczogOTAwCiAgICAgICAgbWF4X3JldHJpZXM6IDEKICAgICAgICBtaW5fcmV0cnlfaW50ZXJ2YWxfbWlsbGlzOiAxMDAwMAogICAgICAtIHRhc2tfa2V5OiBjaG9vc2Vfam9pbgogICAgICAgIGRlcGVuZHNfb246CiAgICAgICAgLSB0YXNrX2tleTogcXVhbGl0eQogICAgICAgIGNvbmRpdGlvbl90YXNrOgogICAgICAgICAgb3A6IEVRVUFMX1RPCiAgICAgICAgICBsZWZ0OiAmI3gyNzt7e2pvYi5wYXJhbWV0ZXJzLmpvaW5fdHlwZX19JiN4Mjc7CiAgICAgICAgICByaWdodDogbGVmdAogICAgICAtIHRhc2tfa2V5OiBsZWZ0X2pvaW4KICAgICAgICBub3RlYm9va190YXNrOgogICAgICAgICAgbm90ZWJvb2tfcGF0aDogLi9zcmMvam9pbi5weQogICAgICAgICAgYmFzZV9wYXJhbWV0ZXJzOgogICAgICAgICAgICBjYXRhbG9nOiAke3Zhci5jYXRhbG9nfQogICAgICAgICAgICBwcmVmaXg6ICR7dmFyLnByZWZpeH0KICAgICAgICAgICAgaG93OiBsZWZ0CiAgICAgICAgdGltZW91dF9zZWNvbmRzOiA5MDAKICAgICAgICBtYXhfcmV0cmllczogMQogICAgICAgIG1pbl9yZXRyeV9pbnRlcnZhbF9taWxsaXM6IDEwMDAwCiAgICAgICAgZGVwZW5kc19vbjoKICAgICAgICAtIHRhc2tfa2V5OiBjaG9vc2Vfam9pbgogICAgICAgICAgb3V0Y29tZTogJiN4Mjc7dHJ1ZSYjeDI3OwogICAgICAtIHRhc2tfa2V5OiBpbm5lcl9qb2luCiAgICAgICAgbm90ZWJvb2tfdGFzazoKICAgICAgICAgIG5vdGVib29rX3BhdGg6IC4vc3JjL2pvaW4ucHkKICAgICAgICAgIGJhc2VfcGFyYW1ldGVyczoKICAgICAgICAgICAgY2F0YWxvZzogJHt2YXIuY2F0YWxvZ30KICAgICAgICAgICAgcHJlZml4OiAke3Zhci5wcmVmaXh9CiAgICAgICAgICAgIGhvdzogaW5uZXIKICAgICAgICB0aW1lb3V0X3NlY29uZHM6IDkwMAogICAgICAgIG1heF9yZXRyaWVzOiAxCiAgICAgICAgbWluX3JldHJ5X2ludGVydmFsX21pbGxpczogMTAwMDAKICAgICAgICBkZXBlbmRzX29uOgogICAgICAgIC0gdGFza19rZXk6IGNob29zZV9qb2luCiAgICAgICAgICBvdXRjb21lOiAmI3gyNztmYWxzZSYjeDI3OwogICAgICAtIHRhc2tfa2V5OiBwdWJsaXNoCiAgICAgICAgbm90ZWJvb2tfdGFzazoKICAgICAgICAgIG5vdGVib29rX3BhdGg6IC4vc3JjL3B1Ymxpc2gucHkKICAgICAgICAgIGJhc2VfcGFyYW1ldGVyczoKICAgICAgICAgICAgY2F0YWxvZzogJHt2YXIuY2F0YWxvZ30KICAgICAgICAgICAgcHJlZml4OiAke3Zhci5wcmVmaXh9CiAgICAgICAgdGltZW91dF9zZWNvbmRzOiA5MDAKICAgICAgICBtYXhfcmV0cmllczogMQogICAgICAgIG1pbl9yZXRyeV9pbnRlcnZhbF9taWxsaXM6IDEwMDAwCiAgICAgICAgZGVwZW5kc19vbjoKICAgICAgICAtIHRhc2tfa2V5OiBsZWZ0X2pvaW4KICAgICAgICAtIHRhc2tfa2V5OiBpbm5lcl9qb2luCiAgICAgICAgcnVuX2lmOiBBVF9MRUFTVF9PTkVfU1VDQ0VTUwogICAgICAtIHRhc2tfa2V5OiBwZXJfY291bnRyeQogICAgICAgIGRlcGVuZHNfb246CiAgICAgICAgLSB0YXNrX2tleTogcHVibGlzaAogICAgICAgIGZvcl9lYWNoX3Rhc2s6CiAgICAgICAgICBpbnB1dHM6ICYjeDI3O1t7JnF1b3Q7Y291bnRyeSZxdW90OzomcXVvdDtJVCZxdW90O30seyZxdW90O2NvdW50cnkmcXVvdDs6JnF1b3Q7RlImcXVvdDt9XSYjeDI3OwogICAgICAgICAgY29uY3VycmVuY3k6IDEKICAgICAgICAgIHRhc2s6CiAgICAgICAgICAgIHRhc2tfa2V5OiBjb3VudHJ5X291dHB1dAogICAgICAgICAgICBub3RlYm9va190YXNrOgogICAgICAgICAgICAgIG5vdGVib29rX3BhdGg6IC4vc3JjL2NvdW50cnkucHkKICAgICAgICAgICAgICBiYXNlX3BhcmFtZXRlcnM6CiAgICAgICAgICAgICAgICBjYXRhbG9nOiAke3Zhci5jYXRhbG9nfQogICAgICAgICAgICAgICAgcHJlZml4OiAke3Zhci5wcmVmaXh9CiAgICAgICAgICAgICAgICBjb3VudHJ5OiAmI3gyNzt7e2lucHV0LmNvdW50cnl9fSYjeDI3OwogICAgICAgICAgICB0aW1lb3V0X3NlY29uZHM6IDkwMAogICAgICAgICAgICBtYXhfcmV0cmllczogMQogICAgICAgICAgICBtaW5fcmV0cnlfaW50ZXJ2YWxfbWlsbGlzOiAxMDAwMAp0YXJnZXRzOgogIGRldjoKICAgIGRlZmF1bHQ6IHRydWUKICAgIG1vZGU6IGRldmVsb3BtZW50CiAgdGVzdDoKICAgIG1vZGU6IHByb2R1Y3Rpb24KICAgIHZhcmlhYmxlczoKICAgICAgcHJlZml4OiBkZWFfcmljY2FyZG9fcHJhY3RpY2VfdGVzdAogIHByb2Q6CiAgICBtb2RlOiBwcm9kdWN0aW9uCiAgICB2YXJpYWJsZXM6CiAgICAgIHByZWZpeDogZGVhX3JpY2NhcmRvX3ByYWN0aWNlX3Byb2QKPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgSm9iIHRhc2sgY291bnRyeS5weTwvc3VtbWFyeT48cHJlIHN0eWxlPSJ3aGl0ZS1zcGFjZTpwcmUtd3JhcDtvdmVyZmxvdy13cmFwOmFueXdoZXJlIj48Y29kZT4jIERhdGFicmlja3Mgbm90ZWJvb2sgc291cmNlCiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL2R5bmFtaWMtdmFsdWUtcmVmZXJlbmNlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy90YXNrLXZhbHVlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9yZWZlcmVuY2UvY2xhc3Nlcy9kYXRhZnJhbWUvam9pbgppbXBvcnQgcmUKZnJvbSBweXNwYXJrLnNxbCBpbXBvcnQgZnVuY3Rpb25zIGFzIEYKZm9yIGtleSwgZGVmYXVsdCBpbiBbKCZxdW90O2NhdGFsb2cmcXVvdDssJnF1b3Q7d29ya3NwYWNlJnF1b3Q7KSwoJnF1b3Q7cHJlZml4JnF1b3Q7LCZxdW90O2RlYV9yaWNjYXJkb19wcmFjdGljZSZxdW90OyldOgogICAgZGJ1dGlscy53aWRnZXRzLnRleHQoa2V5LGRlZmF1bHQpCmNhdGFsb2c9ZGJ1dGlscy53aWRnZXRzLmdldCgmcXVvdDtjYXRhbG9nJnF1b3Q7KTsgcHJlZml4PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7cHJlZml4JnF1b3Q7KQphc3NlcnQgcmUuZnVsbG1hdGNoKHImcXVvdDtbQS1aYS16X11bQS1aYS16MC05X10qJnF1b3Q7LGNhdGFsb2cpCmFzc2VydCByZS5mdWxsbWF0Y2gociZxdW90O1tBLVphLXpfXVtBLVphLXowLTlfXSomcXVvdDsscHJlZml4KQpzaWx2ZXI9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9zaWx2ZXImcXVvdDs7IGdvbGQ9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9nb2xkJnF1b3Q7CgojIENPTU1BTkQgLS0tLS0tLS0tLQpkYnV0aWxzLndpZGdldHMudGV4dCgmcXVvdDtjb3VudHJ5JnF1b3Q7LCZxdW90O0lUJnF1b3Q7KQpjb3VudHJ5PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7Y291bnRyeSZxdW90OykKYXNzZXJ0IGNvdW50cnkgaW4gKCZxdW90O0lUJnF1b3Q7LCZxdW90O0ZSJnF1b3Q7KQooc3BhcmsudGFibGUoZiZxdW90O3tnb2xkfS5qb2JfcmV2ZW51ZSZxdW90OykuZmlsdGVyKEYuY29sKCZxdW90O2NvdW50cnkmcXVvdDspPT1jb3VudHJ5KQogLndyaXRlLm1vZGUoJnF1b3Q7b3ZlcndyaXRlJnF1b3Q7KS5zYXZlQXNUYWJsZShmJnF1b3Q7e2dvbGR9LmpvYl9yZXZlbnVlX3tjb3VudHJ5Lmxvd2VyKCl9JnF1b3Q7KSkKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgSm9iIHRhc2sgam9pbi5weTwvc3VtbWFyeT48cHJlIHN0eWxlPSJ3aGl0ZS1zcGFjZTpwcmUtd3JhcDtvdmVyZmxvdy13cmFwOmFueXdoZXJlIj48Y29kZT4jIERhdGFicmlja3Mgbm90ZWJvb2sgc291cmNlCiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL2R5bmFtaWMtdmFsdWUtcmVmZXJlbmNlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy90YXNrLXZhbHVlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9yZWZlcmVuY2UvY2xhc3Nlcy9kYXRhZnJhbWUvam9pbgppbXBvcnQgcmUKZnJvbSBweXNwYXJrLnNxbCBpbXBvcnQgZnVuY3Rpb25zIGFzIEYKZm9yIGtleSwgZGVmYXVsdCBpbiBbKCZxdW90O2NhdGFsb2cmcXVvdDssJnF1b3Q7d29ya3NwYWNlJnF1b3Q7KSwoJnF1b3Q7cHJlZml4JnF1b3Q7LCZxdW90O2RlYV9yaWNjYXJkb19wcmFjdGljZSZxdW90OyldOgogICAgZGJ1dGlscy53aWRnZXRzLnRleHQoa2V5LGRlZmF1bHQpCmNhdGFsb2c9ZGJ1dGlscy53aWRnZXRzLmdldCgmcXVvdDtjYXRhbG9nJnF1b3Q7KTsgcHJlZml4PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7cHJlZml4JnF1b3Q7KQphc3NlcnQgcmUuZnVsbG1hdGNoKHImcXVvdDtbQS1aYS16X11bQS1aYS16MC05X10qJnF1b3Q7LGNhdGFsb2cpCmFzc2VydCByZS5mdWxsbWF0Y2gociZxdW90O1tBLVphLXpfXVtBLVphLXowLTlfXSomcXVvdDsscHJlZml4KQpzaWx2ZXI9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9zaWx2ZXImcXVvdDs7IGdvbGQ9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9nb2xkJnF1b3Q7CgojIENPTU1BTkQgLS0tLS0tLS0tLQpkYnV0aWxzLndpZGdldHMudGV4dCgmcXVvdDtob3cmcXVvdDssJnF1b3Q7bGVmdCZxdW90OykKaG93PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7aG93JnF1b3Q7KQphc3NlcnQgaG93IGluICgmcXVvdDtsZWZ0JnF1b3Q7LCZxdW90O2lubmVyJnF1b3Q7KQpyZXN1bHQ9c3BhcmsudGFibGUoZiZxdW90O3tzaWx2ZXJ9Lm9yZGVycyZxdW90Oykuam9pbihzcGFyay50YWJsZShmJnF1b3Q7e3NpbHZlcn0uY3VzdG9tZXJzJnF1b3Q7KSwmcXVvdDtjdXN0b21lcl9pZCZxdW90Oyxob3cpCmFzc2VydCByZXN1bHQuY291bnQoKT09KDQgaWYgaG93PT0mcXVvdDtsZWZ0JnF1b3Q7IGVsc2UgMikKcmVzdWx0LndyaXRlLm1vZGUoJnF1b3Q7b3ZlcndyaXRlJnF1b3Q7KS5zYXZlQXNUYWJsZShmJnF1b3Q7e2dvbGR9LmpvYl9qb2luX3Jlc3VsdCZxdW90OykKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgSm9iIHRhc2sgcHVibGlzaC5weTwvc3VtbWFyeT48cHJlIHN0eWxlPSJ3aGl0ZS1zcGFjZTpwcmUtd3JhcDtvdmVyZmxvdy13cmFwOmFueXdoZXJlIj48Y29kZT4jIERhdGFicmlja3Mgbm90ZWJvb2sgc291cmNlCiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL2R5bmFtaWMtdmFsdWUtcmVmZXJlbmNlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy90YXNrLXZhbHVlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9yZWZlcmVuY2UvY2xhc3Nlcy9kYXRhZnJhbWUvam9pbgppbXBvcnQgcmUKZnJvbSBweXNwYXJrLnNxbCBpbXBvcnQgZnVuY3Rpb25zIGFzIEYKZm9yIGtleSwgZGVmYXVsdCBpbiBbKCZxdW90O2NhdGFsb2cmcXVvdDssJnF1b3Q7d29ya3NwYWNlJnF1b3Q7KSwoJnF1b3Q7cHJlZml4JnF1b3Q7LCZxdW90O2RlYV9yaWNjYXJkb19wcmFjdGljZSZxdW90OyldOgogICAgZGJ1dGlscy53aWRnZXRzLnRleHQoa2V5LGRlZmF1bHQpCmNhdGFsb2c9ZGJ1dGlscy53aWRnZXRzLmdldCgmcXVvdDtjYXRhbG9nJnF1b3Q7KTsgcHJlZml4PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7cHJlZml4JnF1b3Q7KQphc3NlcnQgcmUuZnVsbG1hdGNoKHImcXVvdDtbQS1aYS16X11bQS1aYS16MC05X10qJnF1b3Q7LGNhdGFsb2cpCmFzc2VydCByZS5mdWxsbWF0Y2gociZxdW90O1tBLVphLXpfXVtBLVphLXowLTlfXSomcXVvdDsscHJlZml4KQpzaWx2ZXI9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9zaWx2ZXImcXVvdDs7IGdvbGQ9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9nb2xkJnF1b3Q7CgojIENPTU1BTkQgLS0tLS0tLS0tLQpyZXN1bHQ9c3BhcmsudGFibGUoZiZxdW90O3tnb2xkfS5qb2Jfam9pbl9yZXN1bHQmcXVvdDspCihyZXN1bHQud2l0aENvbHVtbigmcXVvdDtjb3VudHJ5JnF1b3Q7LEYuY29hbGVzY2UoJnF1b3Q7Y291bnRyeSZxdW90OyxGLmxpdCgmcXVvdDtVTktOT1dOJnF1b3Q7KSkpLmdyb3VwQnkoJnF1b3Q7Y291bnRyeSZxdW90OykKIC5hZ2coRi5zdW0oJnF1b3Q7YW1vdW50JnF1b3Q7KS5hbGlhcygmcXVvdDtyZXZlbnVlJnF1b3Q7KSkud3JpdGUubW9kZSgmcXVvdDtvdmVyd3JpdGUmcXVvdDspLnNhdmVBc1RhYmxlKGYmcXVvdDt7Z29sZH0uam9iX3JldmVudWUmcXVvdDspKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgSm9iIHRhc2sgcXVhbGl0eS5weTwvc3VtbWFyeT48cHJlIHN0eWxlPSJ3aGl0ZS1zcGFjZTpwcmUtd3JhcDtvdmVyZmxvdy13cmFwOmFueXdoZXJlIj48Y29kZT4jIERhdGFicmlja3Mgbm90ZWJvb2sgc291cmNlCiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9qb2JzL2R5bmFtaWMtdmFsdWUtcmVmZXJlbmNlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy90YXNrLXZhbHVlcwojIGh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9yZWZlcmVuY2UvY2xhc3Nlcy9kYXRhZnJhbWUvam9pbgppbXBvcnQgcmUKZnJvbSBweXNwYXJrLnNxbCBpbXBvcnQgZnVuY3Rpb25zIGFzIEYKZm9yIGtleSwgZGVmYXVsdCBpbiBbKCZxdW90O2NhdGFsb2cmcXVvdDssJnF1b3Q7d29ya3NwYWNlJnF1b3Q7KSwoJnF1b3Q7cHJlZml4JnF1b3Q7LCZxdW90O2RlYV9yaWNjYXJkb19wcmFjdGljZSZxdW90OyldOgogICAgZGJ1dGlscy53aWRnZXRzLnRleHQoa2V5LGRlZmF1bHQpCmNhdGFsb2c9ZGJ1dGlscy53aWRnZXRzLmdldCgmcXVvdDtjYXRhbG9nJnF1b3Q7KTsgcHJlZml4PWRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7cHJlZml4JnF1b3Q7KQphc3NlcnQgcmUuZnVsbG1hdGNoKHImcXVvdDtbQS1aYS16X11bQS1aYS16MC05X10qJnF1b3Q7LGNhdGFsb2cpCmFzc2VydCByZS5mdWxsbWF0Y2gociZxdW90O1tBLVphLXpfXVtBLVphLXowLTlfXSomcXVvdDsscHJlZml4KQpzaWx2ZXI9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9zaWx2ZXImcXVvdDs7IGdvbGQ9ZiZxdW90O3tjYXRhbG9nfS57cHJlZml4fV9nb2xkJnF1b3Q7CgojIENPTU1BTkQgLS0tLS0tLS0tLQpkYnV0aWxzLndpZGdldHMudGV4dCgmcXVvdDtqb2luX3R5cGUmcXVvdDssJnF1b3Q7bGVmdCZxdW90OykKYXNzZXJ0IGRidXRpbHMud2lkZ2V0cy5nZXQoJnF1b3Q7am9pbl90eXBlJnF1b3Q7KSBpbiAoJnF1b3Q7bGVmdCZxdW90OywmcXVvdDtpbm5lciZxdW90OyksICZxdW90O0Nob29zZSBsZWZ0IG9yIGlubmVyJnF1b3Q7Cm9yZGVycz1zcGFyay50YWJsZShmJnF1b3Q7e3NpbHZlcn0ub3JkZXJzJnF1b3Q7KQphc3NlcnQgb3JkZXJzLmNvdW50KCk9PTQsICZxdW90O1J1biBzZXR1cCBhbmQgdHJhbnNmb3JtYXRpb25zIGZvciB0aGlzIHRhcmdldCBmaXJzdCZxdW90Owphc3NlcnQgb3JkZXJzLmZpbHRlcigmcXVvdDthbW91bnQgSVMgTlVMTCBPUiBhbW91bnQgJmx0OyAwJnF1b3Q7KS5jb3VudCgpPT0wCmRidXRpbHMuam9icy50YXNrVmFsdWVzLnNldChrZXk9JnF1b3Q7dmFsaWRfcm93cyZxdW90Oyx2YWx1ZT1vcmRlcnMuY291bnQoKSkKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')
